In [9]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

findspark.init()
spark = SparkSession.builder.master("local[*]").getOrCreate()
df = spark.read.csv('iris.csv', inferSchema=True, header=True)

In [10]:
df = df.withColumnRenamed("sepal.length", "sepal_length")
df = df.withColumnRenamed("sepal.width", "sepal_width")
df = df.withColumnRenamed("petal.length", "petal_length")
df = df.withColumnRenamed("petal.width", "petal_width")
df.head(10)

[Row(sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=4.9, sepal_width=3.0, petal_length=1.4, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=4.7, sepal_width=3.2, petal_length=1.3, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=4.6, sepal_width=3.1, petal_length=1.5, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=5.0, sepal_width=3.6, petal_length=1.4, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=5.4, sepal_width=3.9, petal_length=1.7, petal_width=0.4, variety='Setosa'),
 Row(sepal_length=4.6, sepal_width=3.4, petal_length=1.4, petal_width=0.3, variety='Setosa'),
 Row(sepal_length=5.0, sepal_width=3.4, petal_length=1.5, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=4.4, sepal_width=2.9, petal_length=1.4, petal_width=0.2, variety='Setosa'),
 Row(sepal_length=4.9, sepal_width=3.1, petal_length=1.5, petal_width=0.1, variety='Setosa')]

In [11]:
features = ["sepal_length", 'sepal_width', 'petal_length', 'petal_width']
target =  'variety'
attributes = features + [target]
sample = df.select(attributes).dropna()

In [12]:
assembler = VectorAssembler(inputCols=features, outputCol='features')
output = assembler.transform(sample)
output.show(5)

+------------+-----------+------------+-----------+-------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|variety|         features|
+------------+-----------+------------+-----------+-------+-----------------+
|         5.1|        3.5|         1.4|        0.2| Setosa|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2| Setosa|[4.9,3.0,1.4,0.2]|
|         4.7|        3.2|         1.3|        0.2| Setosa|[4.7,3.2,1.3,0.2]|
|         4.6|        3.1|         1.5|        0.2| Setosa|[4.6,3.1,1.5,0.2]|
|         5.0|        3.6|         1.4|        0.2| Setosa|[5.0,3.6,1.4,0.2]|
+------------+-----------+------------+-----------+-------+-----------------+
only showing top 5 rows


In [13]:
stringIndexer = StringIndexer(inputCol="variety", outputCol="variety_indexed", stringOrderType="frequencyDesc")
output = stringIndexer.fit(output).transform(output)
output.show(5)

+------------+-----------+------------+-----------+-------+-----------------+---------------+
|sepal_length|sepal_width|petal_length|petal_width|variety|         features|variety_indexed|
+------------+-----------+------------+-----------+-------+-----------------+---------------+
|         5.1|        3.5|         1.4|        0.2| Setosa|[5.1,3.5,1.4,0.2]|            0.0|
|         4.9|        3.0|         1.4|        0.2| Setosa|[4.9,3.0,1.4,0.2]|            0.0|
|         4.7|        3.2|         1.3|        0.2| Setosa|[4.7,3.2,1.3,0.2]|            0.0|
|         4.6|        3.1|         1.5|        0.2| Setosa|[4.6,3.1,1.5,0.2]|            0.0|
|         5.0|        3.6|         1.4|        0.2| Setosa|[5.0,3.6,1.4,0.2]|            0.0|
+------------+-----------+------------+-----------+-------+-----------------+---------------+
only showing top 5 rows


In [14]:
train, test = output.randomSplit([0.8, 0.2])

In [15]:
lr = LogisticRegression(featuresCol='features', labelCol='variety_indexed')
model = lr.fit(train)
predictions = model.transform(test)

In [16]:
evaluator = MulticlassClassificationEvaluator(labelCol='variety_indexed')
print('Evaluation:', evaluator.evaluate(predictions))

Evaluation: 0.9721164021164022
